# Building shapes (2).gh Test

## Test Objective
Verify if Building shapes (2).gh file can be called successfully

## Prerequisites
- ✓ File exists: `C:\Users\danie\Downloads\Building shapes (2).gh`
- ⚠️ **Rhino.Compute Service**: Must be running at `http://localhost:6500/`
  - Run `_RhinoCompute` command in Rhino
  - Or start the standalone Rhino.Compute service

## Test Result
The cells below will test the connection with Grasshopper definition


In [1]:
# Check DataTree API
from compute_rhino3d.Grasshopper import DataTree
help(DataTree)

Help on class DataTree in module compute_rhino3d.Grasshopper:

class DataTree(builtins.object)
 |  DataTree(name)
 |  
 |  Methods defined here:
 |  
 |  Append(self, path, items)
 |      Append a path to this tree
 |      
 |      Args:
 |          path (iter): a list of integers defining a path
 |          items (list): list of data to add to the tree
 |  
 |  __init__(self, name)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |  
 |  __dict__
 |      dictionary for instance variables
 |  
 |  __weakref__
 |      list of weak references to the object



In [2]:
# Check compute_rhino3d library documentation
import compute_rhino3d.Grasshopper as gh
help(gh.EvaluateDefinition)

Help on function EvaluateDefinition in module compute_rhino3d.Grasshopper:

EvaluateDefinition(definition, trees)
    Evaluate a grasshopper definition on the compute server.
    
    Args:
        definition (str): path to a grasshopper definition
        trees (iter): list of DataTree instances
    Returns:



In [3]:
%pip install compute-rhino3d -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from pathlib import Path
import traceback

import compute_rhino3d.Util
import compute_rhino3d.Grasshopper as gh
from compute_rhino3d.Grasshopper import DataTree


# =========================
# 1. CONFIG
# =========================

# Rhino.Compute server
compute_rhino3d.Util.url = "http://localhost:6500/"

# Path to your GH file
GH_FILE = r"C:\Users\danie\Downloads\Building shapes (2).gh"


# =========================
# 2. HELPER FUNCTION
# =========================

def make_data_tree(name, value):
    """
    Create a DataTree input for Grasshopper.
    """
    tree = DataTree(name)
    tree.Append([0], [str(value)])
    return tree


def run_grasshopper_definition():
    gh_path = Path(GH_FILE)

    if not gh_path.exists():
        raise FileNotFoundError(f"Grasshopper file not found: {gh_path}")

    # =========================
    # 3. INPUTS TO GH (using DataTree)
    # =========================
    input_names = ["site_area", "edge_count", "north_angle", "shape_type", "tree_count"]
    input_values = [10000, 5, 45, "L", 50]
    
    inputs = [make_data_tree(name, value) for name, value in zip(input_names, input_values)]

    print("✓ Input DataTrees created:")
    for i, name in enumerate(input_names):
        print(f"  {i+1}. {name} = {input_values[i]}")

    # =========================
    # 4. CALL RHINO.COMPUTE
    # =========================
    
    print(f"\n→ Connecting to Rhino.Compute at: {compute_rhino3d.Util.url}")
    print(f"→ Loading Grasshopper definition: {gh_path.name}")
    print(f"→ Full path: {gh_path}")
    
    try:
        # Call the API
        result = gh.EvaluateDefinition(str(gh_path), inputs)
        
        if result is None:
            print("✗ Received empty result from server")
            return None
            
        print("✓ Definition evaluated successfully!")
        
    except json.JSONDecodeError as e:
        print(f"✗ JSON parsing error from server: {e}")
        print(f"  This might indicate:")
        print(f"  - GH definition file has errors")
        print(f"  - Input parameter names don't match GH definition")
        print(f"  - Rhino/GH process issue on server side")
        raise
    except Exception as e:
        print(f"✗ Error type: {type(e).__name__}")
        print(f"✗ Error message: {e}")
        raise

    # =========================
    # 5. PARSE OUTPUTS
    # =========================

    print("\n" + "="*60)
    print("GRASSHOPPER RESULTS")
    print("="*60)
    
    if result is None:
        print("\nNo result returned")
        return None
    
    values = result.get("values", [])
    status = result.get("status", "unknown")
    errors = result.get("errors", [])
    
    print(f"\nStatus: {status}")
    if errors:
        print(f"⚠ Errors: {len(errors)}")
        for error in errors:
            print(f"  - {error}")
    print(f"Number of outputs: {len(values)}")

    if values:
        print("\nOutput Details:")
        for item in values:
            param_name = item.get("ParamName", "unknown")
            inner_tree = item.get("InnerTree", {})
            print(f"\n  Parameter: {param_name}")
            
            for branch, data_list in inner_tree.items():
                print(f"    Branch {branch}: {len(data_list)} items")
                for idx, data in enumerate(data_list):
                    try:
                        if isinstance(data, dict) and "data" in data:
                            raw_data = data.get("data")
                            parsed = json.loads(raw_data) if raw_data else None
                            print(f"      [{idx}]: {parsed}")
                        else:
                            print(f"      [{idx}]: {data}")
                    except Exception as e:
                        print(f"      [{idx}]: (parsing error) {data}")
    else:
        print("\n  (No outputs returned)")

    print("\n" + "="*60)
    return result


# =========================
# 6. RUN
# =========================

if __name__ == "__main__":
    try:
        print("Testing Building shapes (2).gh")
        print("-" * 60)
        result = run_grasshopper_definition()
        if result:
            print("\n✓ SUCCESS: Grasshopper definition evaluated successfully!")
        else:
            print("\n⚠ Completed with no output")
    except FileNotFoundError as e:
        print(f"\n✗ FILE ERROR: {e}")
    except Exception as e:
        print(f"\n✗ ERROR: {type(e).__name__}")
        print(f"\nFull details:")
        traceback.print_exc()


Testing Building shapes (2).gh
------------------------------------------------------------
✓ Input DataTrees created:
  1. site_area = 10000
  2. edge_count = 5
  3. north_angle = 45
  4. shape_type = L
  5. tree_count = 50

→ Connecting to Rhino.Compute at: http://localhost:6500/
→ Loading Grasshopper definition: Building shapes (2).gh
→ Full path: C:\Users\danie\Downloads\Building shapes (2).gh
✗ JSON parsing error from server: Expecting value: line 1 column 1 (char 0)
  This might indicate:
  - GH definition file has errors
  - Input parameter names don't match GH definition
  - Rhino/GH process issue on server side

✗ ERROR: JSONDecodeError

Full details:


Traceback (most recent call last):
  File "c:\Users\danie\anaconda3\envs\genai\Lib\site-packages\requests\models.py", line 978, in json
    return complexjson.loads(self.text, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danie\anaconda3\envs\genai\Lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danie\anaconda3\envs\genai\Lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danie\anaconda3\envs\genai\Lib\json\decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\danie\AppData\Local\Temp\ipykernel_39656\3228646243.py"

In [5]:
# =========================
# Diagnosis: Check GH file input/output parameters
# =========================

from pathlib import Path

GH_FILE = r"C:\Users\danie\Downloads\Building shapes (2).gh"
gh_path = Path(GH_FILE)

print("GH File Information")
print("=" * 60)
print(f"File: {gh_path.name}")
print(f"Exists: {gh_path.exists()}")
print(f"Size: {gh_path.stat().st_size} bytes")
print(f"Path: {gh_path}")

# Try to read the file to see if it's valid
try:
    with open(gh_path, 'rb') as f:
        first_bytes = f.read(100)
        print(f"\nFirst bytes (hex): {first_bytes[:50].hex()}")
        print(f"File signature looks like: binary GH file")
except Exception as e:
    print(f"\nError reading file: {e}")

print("\n" + "-" * 60)
print("Note: GH file content is binary. Cannot read parameter names directly.")
print("Parameters must match exactly with GH definition:")
print("  - Check input parameter names in GH definition")
print("  - Ensure spelling and capitalization match exactly")
print("  - Try with simple/empty input list to see basic response")

GH File Information
File: Building shapes (2).gh
Exists: True
Size: 79519 bytes
Path: C:\Users\danie\Downloads\Building shapes (2).gh

First bytes (hex): ec5d077c5355170f9bb2f71e61cf60f6404647d2455b0a65cfbebcbcd068694bd3b2c18aec59f686822c41a0205356650b8e
File signature looks like: binary GH file

------------------------------------------------------------
Note: GH file content is binary. Cannot read parameter names directly.
Parameters must match exactly with GH definition:
  - Check input parameter names in GH definition
  - Ensure spelling and capitalization match exactly
  - Try with simple/empty input list to see basic response


In [6]:
# =========================
# Test: Call without parameters (check if response is received)
# =========================

import json
import requests
from pathlib import Path
import compute_rhino3d.Util
import compute_rhino3d.Grasshopper as gh
from compute_rhino3d.Grasshopper import DataTree

GH_FILE = r"C:\Users\danie\Downloads\Building shapes (2).gh"
gh_path = Path(GH_FILE)

print("Attempting to call GH file without parameters")
print("=" * 60)

# Attempt 1: Call with no parameters
print("\n[1] Call with no parameters...")
try:
    result = gh.EvaluateDefinition(str(gh_path), [])
    print("✓ Success!")
    print(f"Result: {json.dumps(result, indent=2)[:500]}")
except Exception as e:
    print(f"✗ Failed: {type(e).__name__}: {str(e)[:100]}")

# Attempt 2: Try with one parameter - test common names
print("\n[2] Testing common input parameter names...")
common_names = ["Input", "Geometry", "Data", "Value", "Parameter"]
for param_name in common_names:
    try:
        tree = DataTree(param_name)
        tree.Append([0], ["test"])
        result = gh.EvaluateDefinition(str(gh_path), [tree])
        if result and result.get("values"):
            print(f"✓ '{param_name}' may be valid!")
            break
    except Exception as e:
        pass
else:
    print("✗ Common names not valid")

print("\n" + "=" * 60)
print("Next steps:")
print("  1. Open GH file to check exact input parameter names")
print("  2. Ensure parameter names are spelled correctly (including case)")
print("  3. Check for more information in Rhino console")


Attempting to call GH file without parameters

[1] Call with no parameters...
✗ Failed: JSONDecodeError: Expecting value: line 1 column 1 (char 0)

[2] Testing common input parameter names...
✗ Common names not valid

Next steps:
  1. Open GH file to check exact input parameter names
  2. Ensure parameter names are spelled correctly (including case)
  3. Check for more information in Rhino console


In [7]:
# =========================
# Diagnosis: Check Rhino.Compute service status
# =========================

import requests
import compute_rhino3d.Util

server_url = "http://localhost:6500/"

print("Diagnosing Rhino.Compute service")
print("=" * 60)
print(f"Server URL: {server_url}")

try:
    response = requests.get(server_url, timeout=2)
    print(f"✓ Service connection successful! Status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print(f"✗ Unable to connect to service")
    print(f"\nSolution:")
    print(f"  1. Run '_RhinoCompute' command in Rhino")
    print(f"  2. Or start the standalone Rhino.Compute service")
    print(f"  3. Make sure the service is running at {server_url}")
except Exception as e:
    print(f"✗ Error: {e}")

Diagnosing Rhino.Compute service
Server URL: http://localhost:6500/
✓ Service connection successful! Status code: 200


## Test Summary

### ✓ Successful Items
1. **File exists**: `Building shapes (2).gh` - File size: 79.5 KB
2. **Rhino.Compute connection**: Service running normally (http://localhost:6500/) - HTTP 200
3. **Python API**: `compute-rhino3d` library installed and working
4. **DataTree API**: Data tree objects created correctly

### ✗ Issues
- **GH definition evaluation failed**: Server returned empty JSON response
- **Possible causes**:
  1. Input parameter names don't match GH file definition
  2. GH file has compatibility issues in Rhino.Compute environment
  3. Rhino/GH process needs to be restarted
  4. GH file contains components that cannot run in Compute

### 💡 Recommended Troubleshooting Steps

#### 1. Check GH File Parameters
- Open `Building shapes (2).gh` in Rhino
- Check all green "Input" parameters
- Note the exact parameter names (including case)
- Check if parameters are marked as input/output

#### 2. Create Test Parameter List
```python
# Update based on actual parameters in GH
inputs_map = {
    "actual_param_name_1": value1,
    "actual_param_name_2": value2,
    # ...
}
```

#### 3. Retry the Call
Re-run the test with correct parameter names

#### 4. Check Server Logs
- View Rhino.Compute output/error logs
- May contain detailed error information

### Related Files
- GH definition: `C:\Users\danie\Downloads\Building shapes (2).gh`
- Test script: Cells in this notebook
